In [1]:
# Install necessary packages if not already installed
# !pip install torch torchvision wandb omegaconf

# Import necessary libraries
import os
import json
import time
from pathlib import Path
import torch
import torchinfo
from omegaconf import OmegaConf

import wandb

from dendritic_modeling.training import TrainerMLE, CustomWeightDecayOptimizer
from dendritic_modeling.models import EINetClassifierNLL
from dendritic_modeling.plot_utils import plot_einet_params, plot_einet_activations, plot_voltage_before_activation, plot_activation_tuning
from dendritic_modeling.utils import load_MNIST, split_MNIST_inputs, accuracy_score, save_dict, Shaper

# Set logging
from dendritic_modeling import logger, logger_manager

In [2]:
# Define configurations inline
config = OmegaConf.create({
    "train": {
        "run_name": "dendritic_modeling_sweep",
        "seed": 0,
        "epochs": 50,
        "batch_size": 500,
        "train_valid_split": 0.8,
        "grad_clip_value": 5,
        "lr": 0.01,
    },
    "wandb": {
        "entity": "houman_safaai-harvard-university",
        "project": "dendritic_modeling_pr",
        "group": "debug",
        "tags": ["mnist_1"],
    },
    "model": {
        "input_dim": 784,
        "excitatory_layer_sizes": [10],
        "inhibitory_layer_sizes": [20],
        "excitatory_branch_factors": [1,2,2],
        "inhibitory_branch_factors": [],
        "reactivate": True,
        "somatic_synapses": True,
        "ee_synapses_per_branch_per_layer": [24],
        "ei_synapses_per_branch_per_layer": [100],
        "ie_synapses_per_branch_per_layer": [1],
        "ii_synapses_per_branch_per_layer": [],
        "topk_init_method": "xavier_normal",
    }
})

# Output directory and experiment name
output_dir = "results"
experiment_name = "example"

# Setup W&B
wandb.init(
    project=config.wandb.project,
    entity=config.wandb.entity,
    group=config.wandb.group,
    name=experiment_name,
    config=OmegaConf.to_container(config),
)
print("W&B initialized successfully!")

# Create output directory
os.makedirs(output_dir, exist_ok=True)

# Load MNIST dataset
print("Loading MNIST dataset...")
data = load_MNIST(train_valid_split=config.train.train_valid_split, cache_dir=os.getcwd())
print("MNIST loaded successfully!")



wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: houman_safaai (houman_safaai-harvard-university). Use `wandb login --relogin` to force relogin


W&B initialized successfully!
Loading MNIST dataset...
MNIST loaded successfully!


In [3]:
# Initialize the model
print("Initializing model...")
model_hp = OmegaConf.to_container(config.model)
clf = EINetClassifierNLL(**model_hp)

# Print model summary
model_summary = torchinfo.summary(
    clf,
    input_size=(config.train.batch_size, config.model.input_dim),
    col_names=["input_size", "output_size", "num_params"],
    depth=5,
)
print(model_summary)

# Initialize the trainer
print("Initializing trainer...")
optimizer = CustomWeightDecayOptimizer(
    model = clf, 
    optimizer = torch.optim.Adam(clf.parameters(), lr=config.train.lr),
    weight_decay = 0.1,
)
trainer = TrainerMLE(
    optimizer=optimizer,
    suppress_prints=False,
    print_every=10,
)

Initializing model...
Layer (type:depth-idx)                                       Input Shape               Output Shape              Param #
EINetClassifierNLL                                           [500, 784]                [500, 10]                 10
├─ExcitationInhibitionNetwork: 1-1                           [500, 784]                [500, 10]                 --
│    └─ModuleList: 2-1                                       --                        --                        --
│    │    └─ExcitationInhibitionLayer: 3-1                   [500, 784]                [500, 10]                 --
│    │    │    └─DendriNet: 4-1                              [500, 784]                [500, 20]                 --
│    │    │    │    └─DendriticBranchLayer: 5-1              [500, 784]                [500, 20]                 15,720
│    │    │    └─DendriNet: 4-2                              [500, 784]                [500, 10]                 --
│    │    │    │    └─ModuleList: 5-2    

In [4]:
print("Starting model training...")

start_time = time.time()
trainer.train(
    model=clf,
    train_data=data['train'],
    valid_data=data['valid'],
    epochs=config.train.epochs,
    batch_size=config.train.batch_size,
    grad_clip_value=config.train.grad_clip_value,
    shuffle=True,
    plot_losses=True,
    save_path=output_dir,
)
training_time = time.time() - start_time
print(f"Training completed in {training_time:.2f} seconds.")

Starting model training...
INFO training.py:312   ----   Epoch 10  ---- 
INFO training.py:313  training loss: 0.27 | validation loss: 0.27
INFO training.py:319  time elapsed: 116.02 s | predicted time remaining: 464.08 s
INFO training.py:312   ----   Epoch 20  ---- 
INFO training.py:313  training loss: 0.24 | validation loss: 0.27
INFO training.py:319  time elapsed: 228.76 s | predicted time remaining: 343.14 s
INFO training.py:312   ----   Epoch 30  ---- 
INFO training.py:313  training loss: 0.23 | validation loss: 0.24
INFO training.py:319  time elapsed: 341.73 s | predicted time remaining: 227.82 s
INFO training.py:312   ----   Epoch 40  ---- 
INFO training.py:313  training loss: 0.22 | validation loss: 0.22
INFO training.py:319  time elapsed: 454.55 s | predicted time remaining: 113.64 s
INFO training.py:312   ----   Epoch 50  ---- 
INFO training.py:313  training loss: 0.21 | validation loss: 0.22
INFO training.py:319  time elapsed: 567.10 s | predicted time remaining: 0.00 s
Train

In [5]:
# Evaluate performance
print("Evaluating performance...")
performance = {
    'train accuracy': accuracy_score(clf, *data['train'][:]).item(),
    'valid accuracy': accuracy_score(clf, *data['valid'][:]).item(),
    'test accuracy': accuracy_score(clf, *data['test'][:]).item(),
}
performance['training time'] = training_time
print("Performance:", performance)

# Save results
save_dict(model_hp, output_dir, 'model_hparams.json')
save_dict(performance, output_dir, 'performance.json')
print("Results saved!")

Evaluating performance...
Performance: {'train accuracy': 0.9408957958221436, 'valid accuracy': 0.9383333325386047, 'test accuracy': 0.9350000023841858, 'training time': 567.6544442176819}
Results saved!


In [6]:
# Visualize model parameters and activations
print("Visualizing parameters and activations...")
shaper = Shaper(shape=(28, 28))

plot_einet_params(
    einet=clf.net, save_root=os.path.join(output_dir, 'param_visuals'),
    reshape_fn=shaper.reshape, reshape_exc_syn=True, logspace = True,
)

input_list = split_MNIST_inputs(*data['train'][:])
plot_einet_activations(
    einet=clf.net, input_list=input_list,
    save_root=os.path.join(output_dir, 'activation_visuals'),
)

# # Plot voltage before activation
# plot_voltage_before_activation(
#     einet=clf.net,  # Your trained model
#     input_list=input_list,
#     save_root="voltage_plots"  # Directory to save plots
# )




Visualizing parameters and activations...


In [7]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Data transformation for MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # Mean and std for MNIST
])

# Load MNIST Dataset
train_data = datasets.MNIST(root="./data", train=True, transform=transform, download=True)
train_loader = DataLoader(train_data, batch_size=100, shuffle=True)


 
# Prepare input tensors for digits 0-9
def get_inputs_per_digit(train_loader, num_samples_per_digit=100):
    """
    Collects input samples for each digit (0-9) from the training loader.

    Args:
        train_loader: DataLoader for training data.
        num_samples_per_digit (int): Number of samples per digit to collect.

    Returns:
        list: A list where each element corresponds to a tensor of inputs for one digit.
    """
    inputs_per_digit = {digit: [] for digit in range(10)}  # Dictionary to store inputs by digit
    
    for images, labels in train_loader:
        images = images.view(images.size(0), -1)  # Flatten images if necessary
        for i in range(images.size(0)):
            digit = labels[i].item()
            if len(inputs_per_digit[digit]) < num_samples_per_digit:
                inputs_per_digit[digit].append(images[i])
        
        # Stop once we have enough samples for each digit
        if all(len(inputs) >= num_samples_per_digit for inputs in inputs_per_digit.values()):
            break

    # Convert lists to tensors
    input_list = [torch.stack(inputs_per_digit[digit]) for digit in range(10)]
    return input_list

# Assuming `train_loader` is your training data loader
input_list = get_inputs_per_digit(train_loader, num_samples_per_digit=100)

# Save the tuning plots 
save_root=os.path.join(output_dir, 'activation_tuning_plots')
#save_root = "activation_tuning_plots"
plot_activation_tuning(
    einet=clf.net,  # Your trained model
    input_list=input_list,
    save_root=save_root
)
print("Activation tuning plots saved successfully.")

print("Visualization completed.")

Saved tuning plot for layer1_inh at results/activation_tuning_plots/layer1_inh_tuning.png
Saved tuning plot for layer1_exc at results/activation_tuning_plots/layer1_exc_tuning.png
Activation tuning plots saved successfully.
Visualization completed.


In [8]:
wandb.finish()
print("Experiment finished successfully!")

epoch,▁▁▁▁▁▂▂▂▂▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇████
train_loss,█▄▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_accuracy,▁▄▅▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇█▇▇▇███████████████
val_f1_score,▁▄▅▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇█▇▇▇▇██████████████
val_precision,▁▄▅▆▆▇▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇█▇▇▇██████████████
val_recall,▁▄▅▆▆▇▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇█▇▇▇▇█████████████
valid_loss,█▄▃▂▂▂▂▂▂▁▁▁▁▁▁▁▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,50
train_loss,0.21462
val_accuracy,0.93383
val_f1_score,0.93404


Experiment finished successfully!
